<p> <center><img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:200px"> </p>

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Analyse et prédiction de la variabilité de la production solaire à partir de données ouvertes de la région PACA </h1></center>
<center><h2> Features Engineering </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

# I - Imports et récupération des données précédentes


Nous commencons par importer les librairies nécessaires pour manipuler nos données :

In [1]:
# Gestion des chemins
from pathlib import Path

# Données et calculs
import pandas as pd
import numpy as np


On récupère les données collectées aux étapes précédentes :

In [2]:
# Chemin vers le répertoire de données d'entrée
input_path = Path('../../data/local_data/input/')

# Chemin vers le répertoire de résultats temporaires
temp_path = Path('../../data/local_data/temp/')

# Chemin vers le répertoire de résultats finaux
output_path = Path('../../data/local_data/output/')

# Chemin du dataset de production
input_datasets = output_path / 'raw_2020_2025.csv'


In [3]:
# Lecture du fichiers enregistrés à l'étape précédente
all_df = {
    'train' : pd.read_csv(temp_path / ('raw_train_5pts.csv'), index_col='datetime_utc', parse_dates=True),
    'valid' : pd.read_csv(temp_path / ('raw_valid_5pts.csv'), index_col='datetime_utc', parse_dates=True),
    'test' : pd.read_csv(temp_path / ('raw_test_5pts.csv'), index_col='datetime_utc', parse_dates=True)}

for key, df in all_df.items():
    print(f'\n{key} :')
    display(df.head())



train :


,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,...,bra_humidite,eyg_temperature,eyg_vitesse_vent,eyg_nebulosite,eyg_humidite,target,sin_hour,cos_hour,sin_doy,cos_doy
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2019-12-31 23:00:00+00:00,6123.0,0.0,0.0,0.0,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,...,86.880,2.720,1.220,51.750,100.000,0.0,-0.258819,0.965926,-0.017213,0.999852
2019-12-31 23:30:00+00:00,5907.0,0.0,0.0,0.0,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,...,84.455,2.760,1.405,54.215,98.790,0.0,-0.130526,0.991445,-0.017213,0.999852
2020-01-01 00:00:00+00:00,5724.0,0.0,0.0,0.0,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,...,82.030,2.800,1.590,56.680,97.580,0.0,0.000000,1.000000,0.000000,1.000000
2020-01-01 00:30:00+00:00,5749.0,0.0,0.0,0.0,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,...,79.380,2.675,1.635,33.325,96.065,0.0,0.130526,0.991445,0.000000,1.000000
2020-01-01 01:00:00+00:00,5700.0,0.0,0.0,0.0,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,...,76.730,2.550,1.680,9.970,94.550,0.0,0.258819,0.965926,0.000000,1.000000



valid :


,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,...,bra_humidite,eyg_temperature,eyg_vitesse_vent,eyg_nebulosite,eyg_humidite,target,sin_hour,cos_hour,sin_doy,cos_doy
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,4815.0,0.0,0.0,0.0,12.879512,-68.564229,13.426722,-68.099906,15.632999,-68.757915,...,95.680,2.710,1.440,99.410,96.430,0.0,0.000000,1.000000,0.0,1.0
2024-01-01 00:30:00+00:00,4853.0,0.0,0.0,0.0,30.257376,-66.571182,30.448105,-66.089748,32.855587,-66.517697,...,94.865,2.620,1.395,99.705,96.195,0.0,0.130526,0.991445,0.0,1.0
2024-01-01 01:00:00+00:00,4834.0,0.0,0.0,0.0,44.628236,-63.284768,44.591334,-62.821838,46.880350,-63.031066,...,94.050,2.530,1.350,100.000,95.960,0.0,0.258819,0.965926,0.0,1.0
2024-01-01 01:30:00+00:00,4964.0,0.0,0.0,0.0,56.102397,-59.130058,55.958833,-58.702637,57.998488,-58.727575,...,93.455,2.535,1.290,99.585,95.500,0.0,0.382683,0.923880,0.0,1.0
2024-01-01 02:00:00+00:00,4791.0,0.0,0.0,0.0,65.344202,-54.429271,65.169535,-54.042984,66.942140,-53.918107,...,92.860,2.540,1.230,99.170,95.040,0.0,0.500000,0.866025,0.0,1.0



test :


,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,...,bra_humidite,eyg_temperature,eyg_vitesse_vent,eyg_nebulosite,eyg_humidite,target,sin_hour,cos_hour,sin_doy,cos_doy
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00+00:00,5492.0,0.0,0.0,0.0,12.621973,-68.519484,13.173648,-68.055736,15.368659,-68.716819,...,89.010,3.26,1.000,96.43,93.350,0.0,0.000000,1.000000,0.0,1.0
2025-01-01 00:30:00+00:00,5498.0,0.0,0.0,0.0,30.001046,-66.548921,30.196673,-66.067425,32.597940,-66.499127,...,89.245,3.18,1.000,97.87,93.565,0.0,0.130526,0.991445,0.0,1.0
2025-01-01 01:00:00+00:00,5424.0,0.0,0.0,0.0,44.396127,-63.280642,44.362936,-62.817182,46.650081,-63.030109,...,89.480,3.10,1.000,99.31,93.780,0.0,0.258819,0.965926,0.0,1.0
2025-01-01 01:30:00+00:00,5506.0,0.0,0.0,0.0,55.899419,-59.138851,55.758234,-58.710650,57.798329,-58.738902,...,89.010,3.09,0.995,99.64,93.325,0.0,0.382683,0.923880,0.0,1.0
2025-01-01 02:00:00+00:00,5285.0,0.0,0.0,0.0,65.166910,-54.446690,64.993649,-54.059517,66.767626,-53.937514,...,88.540,3.08,0.990,99.97,92.870,0.0,0.500000,0.866025,0.0,1.0


# II - Ajout de variables

A l'étape précédente, nous avons ajouté quelques variables, concernant l'**encodage cyclique de la date** (`sin_hour`, `cos_hour`, `sin_doy`, `cos_doy`) et le calcul de la **variable cible** (`target`) à partir de la variable `tch_solaire` aux instants t+1 et t.

Nous avons cependant besoin d'autres variables pour permettre au modèle d'intégrer l'évolution temporelle de nos variables. 

On souhaite réaliser des prédictions à 30 minutes : on s'intéresse particulièrement aux évolutions récentes des variables, dans les heures précédentes (par exemple 2h ou 4h).

In [4]:
# Nombre de pas de temps correspondant aux fenêtres
window_2h = 4   # 2 heures = 4 pas de 30 minutes
window_4h = 8   # 4 heures = 8 pas de 30 minutes

## A - Irradiance solaire

**BNI – Beam (Direct) Normal Irradiance**  
Le $BNI$ correspond au rayonnement solaire direct mesuré sur un plan perpendiculaire aux rayons du soleil.  
Sur un plan horizontal, sa contribution s'écrit :
$$
BHI = BNI \cdot \cos(\theta_z)
$$
où $\theta_z = \dfrac{\pi}{2} - \alpha$ est l'angle zénithal solaire et $\alpha$ l'**altitude solaire**, définie comme l'angle entre le soleil et l'horizon.  
Le $BNI$ dépend donc fortement de la position instantanée du soleil et suppose une orientation optimale du capteur. Or, les panneaux photovoltaïques fixes ne sont pas toujours perpendiculaires aux rayons solaires et ne suivent généralement pas le soleil.

**GHI – Global Horizontal Irradiance**  
Le $GHI$ représente le rayonnement solaire total reçu sur un plan horizontal et s'exprime par :
$$
GHI = BHI + DHI = BNI \cdot \cos(\theta_z) + DHI
$$
où $DHI$ est l'irradiance diffuse horizontale.  
Le $GHI$ intègre ainsi les composantes directe et diffuse du rayonnement et décrit plus fidèlement l'irradiation réellement reçue par les panneaux photovoltaïques standards.

**Choix méthodologique**  
Dans notre projet, nous adoptons le **GHI** comme variable d'entrée principale, car il permet une analyse plus représentative et plus robuste de la variabilité de la production solaire à partir de données ouvertes.

On considère donc les variations temporelles de l'irradiance solaire :

$$
\Delta GHI(t) = GHI(t) - GHI(t-\Delta t)
$$
avec $\Delta t = 30$ minutes.


In [5]:
# GHI
for key, df in all_df.items():
    for ville in ['bra', 'cru', 'eyg', 'sel', 'svt'] :
        # Variation de l'irradiance
        df[ville+"_dghi_dt"] = df[ville+"_ghi"].diff()

        # Lags de l'irradiance
        df[ville+"_ghi_lag_1"] = df[ville+"_ghi"].shift(1)
        df[ville+"_ghi_lag_2"] = df[ville+"_ghi"].shift(2)

        # Lags de la variation de l'irradiance
        df[ville+"_dghi_lag_1"] = df[ville+"_dghi_dt"].shift(1)
        df[ville+"_dghi_lag_2"] = df[ville+"_dghi_dt"].shift(2)
        df[ville+"_dghi_lag_3"] = df[ville+"_dghi_dt"].shift(3)

        # Moyennes glissantes
        df[ville+"_ghi_mean_2h"] = df[ville+"_ghi"].rolling(window=window_2h).mean()
        df[ville+"_ghi_mean_4h"] = df[ville+"_ghi"].rolling(window=window_4h).mean()

        # Ecarts-type glissants
        df[ville+"_ghi_std_2h"] = df[ville+"_ghi"].rolling(window=window_2h).std()
        df[ville+"_ghi_std_4h"] = df[ville+"_ghi"].rolling(window=window_4h).std()

    print(f"Taille de {key} -> {df.shape}")

Taille de train -> (70130, 139)
Taille de valid -> (17568, 139)
Taille de test -> (17520, 139)


## B - Nébulosité

Les rampes de production photovoltaïque sont principalement déclenchées par des variations rapides de l'irradiance solaire, généralement associées aux passages nuageux.

On considère donc les variations temporelles de la nébulosité :

$$
\Delta \text{Nebulosite}(t) =
\text{Nebulosite}(t) -
\text{Nebulosite}(t-\Delta t)
$$
avec $\Delta t = 30$ minutes.

Ces variables capturent directement les transitions nuageuses responsables des rampes de production photovoltaïque.


In [6]:
# Nébulosité
for key, df in all_df.items():
    for ville in ['bra', 'cru', 'eyg', 'sel', 'svt'] :
        # Variation de la nébulosité
        df[ville+"_dnebulosite_dt"] = df[ville+"_nebulosite"].diff()

        # Lags de la nébulosité
        df[ville+"_nebulosite_lag_1"] = df[ville+"_nebulosite"].shift(1)
        df[ville+"_nebulosite_lag_2"] = df[ville+"_nebulosite"].shift(2)

        # Lags de la variation de la nébulosité
        df[ville+"_dnebulosite_lag_1"] = df[ville+"_dnebulosite_dt"].shift(1)
        df[ville+"_dnebulosite_lag_2"] = df[ville+"_dnebulosite_dt"].shift(2)

    print(f"Taille de {key} : {df.shape}")

Taille de train : (70130, 164)
Taille de valid : (17568, 164)
Taille de test : (17520, 164)


## C - Production d'énergie

On considère également la production des panneaux photovoltaïques au cours des heures précédentes, ainsi que les valeurs précédentes de notre variable cible.

In [7]:
# TCH solaire
for key, df in all_df.items():
    # Lags du TCH solaire
    df["tch_lag_1"] = df["tch_solaire"].shift(1)
    df["tch_lag_2"] = df["tch_solaire"].shift(2)
    df["tch_lag_3"] = df["tch_solaire"].shift(3)

    # Lags de la variation du TCH solaire
    df["target_lag_1"] = df["target"].shift(1)
    df["target_lag_2"] = df["target"].shift(2)
    df["target_lag_3"] = df["target"].shift(3)

    # Moyennes glissantes (calculées sur tch_lag_1 pour éviter la fuite de données)
    df["tch_mean_2h"] = df["tch_lag_1"].rolling(window=window_2h).mean()
    df["tch_mean_4h"] = df["tch_lag_1"].rolling(window=window_4h).mean()

    # Ecarts-type glissants (calculés sur tch_lag_1 pour éviter la fuite de données)
    df["tch_std_2h"] = df["tch_lag_1"].rolling(window=window_2h).std()
    df["tch_std_4h"] = df["tch_lag_1"].rolling(window=window_4h).std()

    print(f"Taille de {key} : {df.shape}")

Taille de train : (70130, 174)
Taille de valid : (17568, 174)
Taille de test : (17520, 174)


# III - Ordonnancement des jeux de données

Pour faciliter la lecture des jeux de données, on ordonne le nom des colonnes par ordre aplphabétique :

In [8]:
for key, df in all_df.items():
    df = df.reindex(columns=df.columns.sort_values())
    print(f"Taille de {key} : {df.shape}")
    display(df.head())

Taille de train : (70130, 174)


,bra_altitude,bra_azimuth,bra_bhi,bra_bni,bra_clear_sky_bhi,bra_clear_sky_bni,bra_clear_sky_dhi,bra_clear_sky_ghi,bra_dghi_dt,bra_dghi_lag_1,...,target_lag_3,tch_lag_1,tch_lag_2,tch_lag_3,tch_mean_2h,tch_mean_4h,tch_solaire,tch_std_2h,tch_std_4h,tco_solaire
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2019-12-31 23:00:00+00:00,-68.046190,335.241040,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2019-12-31 23:30:00+00:00,-69.500739,353.945722,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2020-01-01 00:00:00+00:00,-69.141123,13.536657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2020-01-01 00:30:00+00:00,-67.054165,31.240402,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN,0.0
2020-01-01 01:00:00+00:00,-63.656435,45.699703,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0


Taille de valid : (17568, 174)


,bra_altitude,bra_azimuth,bra_bhi,bra_bni,bra_clear_sky_bhi,bra_clear_sky_bni,bra_clear_sky_dhi,bra_clear_sky_ghi,bra_dghi_dt,bra_dghi_lag_1,...,target_lag_3,tch_lag_1,tch_lag_2,tch_lag_3,tch_mean_2h,tch_mean_4h,tch_solaire,tch_std_2h,tch_std_4h,tco_solaire
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,-69.141088,13.532368,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2024-01-01 00:30:00+00:00,-67.054481,31.236592,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2024-01-01 01:00:00+00:00,-63.657001,45.696549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2024-01-01 01:30:00+00:00,-59.396982,57.119939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN,0.0
2024-01-01 02:00:00+00:00,-54.603773,66.256046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0


Taille de test : (17520, 174)


,bra_altitude,bra_azimuth,bra_bhi,bra_bni,bra_clear_sky_bhi,bra_clear_sky_bni,bra_clear_sky_dhi,bra_clear_sky_ghi,bra_dghi_dt,bra_dghi_lag_1,...,target_lag_3,tch_lag_1,tch_lag_2,tch_lag_3,tch_mean_2h,tch_mean_4h,tch_solaire,tch_std_2h,tch_std_4h,tco_solaire
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00+00:00,-69.097348,13.266719,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2025-01-01 00:30:00+00:00,-67.033957,30.974224,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2025-01-01 01:00:00+00:00,-63.654884,45.461256,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0
2025-01-01 01:30:00+00:00,-59.407753,56.915724,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN,0.0
2025-01-01 02:00:00+00:00,-54.623026,66.078554,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0


# IV - Suppression des valeurs nulles

La création des nouvelles variables a introduit des valeurs nulles dans les jeux de données : on supprime les lignes correspondantes.

In [9]:
# Suppression des valeurs nulles introduites
for key, df in all_df.items():
    print(f"Taille de {key} : {df.shape}")
    df.dropna(inplace=True)
    print(f"Taille de {key} : {df.shape}")

Taille de train : (70130, 174)
Taille de train : (70121, 174)
Taille de valid : (17568, 174)
Taille de valid : (17559, 174)
Taille de test : (17520, 174)
Taille de test : (17511, 174)


# V - Enregistrement des résultats

In [10]:
# Sauvegarde des datasets 
for key, df in all_df.items():
    df.to_csv(temp_path / (key + '_5pts.csv'))
    

In [11]:
# Vérification
df_train = pd.read_csv(
    temp_path / ('train_5pts.csv'), 
    index_col='datetime_utc', 
    parse_dates=True)
display(df_train)


df_valid = pd.read_csv(
    temp_path / ('valid_5pts.csv'), 
    index_col='datetime_utc', 
    parse_dates=True)
display(df_valid)


df_test = pd.read_csv(
    temp_path / ('test_5pts.csv'), 
    index_col='datetime_utc', 
    parse_dates=True)
display(df_test)


,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,...,tch_lag_1,tch_lag_2,tch_lag_3,target_lag_1,target_lag_2,target_lag_3,tch_mean_2h,tch_mean_4h,tch_std_2h,tch_std_4h
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2020-01-01 03:00:00+00:00,5332.0,0.0,0.0,0.0,79.584354,-44.161912,79.447862,-43.858394,80.769679,-43.509092,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 03:30:00+00:00,5219.0,0.0,0.0,0.0,85.424299,-38.823177,85.326319,-38.559425,86.472049,-38.123002,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 04:00:00+00:00,5157.0,0.0,0.0,0.0,90.773227,-33.441532,90.717934,-33.216659,91.715185,-32.704136,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 04:30:00+00:00,5161.0,0.0,0.0,0.0,95.807765,-28.065765,95.796716,-27.879311,96.668578,-27.299031,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 05:00:00+00:00,5108.0,0.0,0.0,0.0,100.660015,-22.736496,100.693455,-22.588455,101.459404,-21.946957,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 21:00:00+00:00,4752.0,0.0,0.0,0.0,289.546935,-51.221525,290.365261,-51.293713,290.103312,-52.038109,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-12-31 21:30:00+00:00,4827.0,0.0,0.0,0.0,297.707267,-56.154878,298.650377,-56.158211,298.413679,-56.979503,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-12-31 22:00:00+00:00,4982.0,0.0,0.0,0.0,307.663811,-60.689554,308.733106,-60.609034,308.632681,-61.500872,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,...,tch_lag_1,tch_lag_2,tch_lag_3,target_lag_1,target_lag_2,target_lag_3,tch_mean_2h,tch_mean_4h,tch_std_2h,tch_std_4h
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2024-01-01 04:00:00+00:00,4358.0,0.0,0.00,0.00,90.771843,-33.442421,90.716545,-33.217538,91.713821,-32.705034,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 04:30:00+00:00,4359.0,0.0,0.00,0.00,95.806454,-28.066637,95.795398,-27.880173,96.667282,-27.299910,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 05:00:00+00:00,4302.0,0.0,0.00,0.00,100.658750,-22.737341,100.692183,-22.589290,101.458151,-21.947808,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 05:30:00+00:00,4305.0,0.0,0.00,0.00,105.432107,-17.491337,105.509585,-17.382122,106.186495,-16.684796,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 06:00:00+00:00,4313.0,0.0,0.00,0.00,110.211322,-12.364522,110.331968,-12.294953,110.934961,-11.546493,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 21:00:00+00:00,5495.0,0.0,0.00,0.00,289.524863,-51.118559,290.340974,-51.190939,290.081380,-51.935098,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2024-12-31 21:30:00+00:00,5556.0,0.0,0.00,0.00,297.670944,-56.052910,298.611298,-56.056561,298.376539,-56.877538,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2024-12-31 22:00:00+00:00,5638.0,0.0,0.00,0.00,307.602765,-60.589925,308.668798,-60.509924,308.568947,-61.401403,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0


,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,...,tch_lag_1,tch_lag_2,tch_lag_3,target_lag_1,target_lag_2,target_lag_3,tch_mean_2h,tch_mean_4h,tch_std_2h,tch_std_4h
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2025-01-01 04:00:00+00:00,4860.0,0.0,0.0,0.0,90.650757,-33.470475,90.595478,-33.244692,91.594006,-32.733866,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-01-01 04:30:00+00:00,4861.0,0.0,0.0,0.0,95.691654,-28.093986,95.680530,-27.906632,96.553472,-27.327866,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-01-01 05:00:00+00:00,4807.0,0.0,0.0,0.0,100.548392,-22.763105,100.581707,-22.614171,101.348542,-21.974031,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-01-01 05:30:00+00:00,4805.0,0.0,0.0,0.0,105.324719,-17.514720,105.402056,-17.404626,106.079648,-16.708503,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-01-01 06:00:00+00:00,4856.0,0.0,0.0,0.0,110.105716,-12.384770,110.226218,-12.314322,110.829714,-11.566934,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 21:00:00+00:00,5673.0,0.0,0.0,0.0,289.529078,-51.149204,290.345844,-51.221547,290.085511,-51.965752,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-31 21:30:00+00:00,5722.0,0.0,0.0,0.0,297.679116,-56.083362,298.620292,-56.086941,298.384896,-56.907989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-31 22:00:00+00:00,5775.0,0.0,0.0,0.0,307.617966,-60.619832,308.684985,-60.539702,308.584871,-61.431271,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
